# F02_02 — Benchmarks y evaluación de forecasting

## Propósito de la sesión

En la sesión anterior caracterizamos la estructura temporal de una serie. Ahora daremos el siguiente paso: **producir pronósticos simples y evaluar su desempeño fuera de muestra**.

La pregunta central será:

> **¿Qué tan bien puede pronosticarse la serie usando reglas simples y qué referencia deberá superar cualquier modelo posterior?**

Al finalizar la sesión, el estudiante deberá poder:

1. distinguir entre entrenamiento y prueba;
2. definir un horizonte de pronóstico $h$;
3. construir benchmarks mean, naive y drift;
4. calcular errores fuera de muestra;
5. interpretar MAE, RMSE y MASE;
6. comparar métodos usando la misma partición temporal;
7. seleccionar un benchmark razonable para modelos posteriores.

### Cómo trabajar este notebook

- 📘 **Concepto**
- 🌍 **Contexto**
- 💻 **Entendamos el código**
- 🧠 **Predice**
- ▶ **Ejecuta**
- 🔎 **Interpreta**
- ▶ **Ver respuesta razonada**
- 🛠️ **Modifica**


## 1. Preparación del notebook

Cada notebook del curso es **autocontenido**. `F00_configuracion_forecasting.ipynb` verifica el entorno, pero sus imports y variables no se comparten automáticamente.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("data/fpppy")
assert DATA_DIR.exists(), "No se encuentra data/fpppy."

sns.set_style("whitegrid")


## 2. Evaluación fuera de muestra

### 📘 Concepto

Para evaluar forecasting debemos imitar la situación real:

1. usar una parte inicial de la serie como **train**;
2. reservar observaciones posteriores como **test**;
3. construir el pronóstico usando solo `train`;
4. comparar el pronóstico con `test`.

En series temporales **no debemos hacer una partición aleatoria**.

- **train**: datos disponibles al momento de pronosticar;
- **test**: datos futuros reservados para evaluación;
- $h$: horizonte de pronóstico.


### 🌍 Contexto — Ansett

Usaremos la misma serie de Ansett Australia de la Semana 1, pero con un objetivo diferente:

- Ruta: MEL–SYD.
- Clase: Economy.
- Frecuencia: semanal.
- Variable: número de pasajeros.

Ahora no analizaremos principalmente la dependencia temporal, sino la **capacidad de pronóstico fuera de muestra**.


In [ ]:
ansett = pd.read_csv(DATA_DIR / "ansett.csv", parse_dates=["ds"])

serie = (
    ansett
    .loc[
        (ansett["Airports"] == "MEL-SYD")
        & (ansett["Class"] == "Economy"),
        ["ds", "y"]
    ]
    .sort_values("ds")
    .reset_index(drop=True)
)

serie.head()


### 🧠 Predice

Si reservamos las últimas 12 semanas como `test`:

1. ¿qué debe quedar en `train`?
2. ¿sería correcto escoger 12 semanas al azar?
3. ¿qué significa $h=12$?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

1. Todas las observaciones anteriores a las últimas 12.
2. No, porque debemos respetar el orden temporal.
3. Pronosticar 12 semanas hacia adelante.

</details>


In [ ]:
h = 12

train = serie.iloc[:-h].copy()
test = serie.iloc[-h:].copy()

print(f"Total: {len(serie)}")
print(f"Train: {len(train)}")
print(f"Test: {len(test)}")
print(f"Horizonte: {h} semanas")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(train["ds"], train["y"], label="Train")
ax.plot(test["ds"], test["y"], marker="o", label="Test")

ax.set(
    title="Partición temporal train/test",
    xlabel="Fecha",
    ylabel="Pasajeros"
)
ax.legend()
plt.show()


### 🔎 Interpreta

- ¿`test` ocurre completamente después de `train`?
- ¿por qué esta partición representa mejor un problema real de forecasting que una división aleatoria?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

Sí. `test` contiene únicamente observaciones posteriores al entrenamiento. Esto reproduce la situación real: conocemos el pasado, pero no el futuro.

</details>


### 🛠️ Modifica

Cambie temporalmente:

```python
h = 8
```

y reconstruya `train` y `test`. Observe cómo cambia la frontera temporal. Luego regrese a `h = 12`.


## 3. Benchmark mean

### 📘 Concepto

El método de la media pronostica todos los valores futuros con la media histórica:

$$
\hat{y}_{T+h|T}=\bar{y}
$$

Ignora el orden reciente y utiliza solamente el nivel promedio del entrenamiento.


In [ ]:
media_train = train["y"].mean()

forecast_mean = pd.DataFrame({
    "ds": test["ds"].to_numpy(),
    "y_hat": np.repeat(media_train, h)
})

forecast_mean.head()


## 4. Benchmark naive

### 📘 Concepto

El método naive usa la última observación conocida:

$$
\hat{y}_{T+h|T}=y_T
$$

Es una referencia fundamental cuando existe persistencia temporal de corto plazo.


In [ ]:
ultimo_valor = train["y"].iloc[-1]

forecast_naive = pd.DataFrame({
    "ds": test["ds"].to_numpy(),
    "y_hat": np.repeat(ultimo_valor, h)
})

forecast_naive.head()


### 💻 Entendamos el código

```python
train["y"].iloc[-1]
```

selecciona la última observación del entrenamiento.

```python
np.repeat(ultimo_valor, h)
```

repite ese valor una vez por cada período futuro.


## 5. Benchmark drift

### 📘 Concepto

Drift prolonga el cambio promedio observado entre el primer y el último dato:

$$
\hat{y}_{T+h|T}
=
y_T+h\left(\frac{y_T-y_1}{T-1}\right)
$$

A diferencia de mean y naive, su pronóstico cambia con el horizonte.


In [ ]:
y_1 = train["y"].iloc[0]
y_T = train["y"].iloc[-1]
T = len(train)

drift_por_semana = (y_T - y_1) / (T - 1)
pasos = np.arange(1, h + 1)

forecast_drift = pd.DataFrame({
    "ds": test["ds"].to_numpy(),
    "y_hat": y_T + pasos * drift_por_semana
})

forecast_drift.head()


## 6. Comparación visual de benchmarks


In [ ]:
comparacion = pd.DataFrame({
    "ds": test["ds"].to_numpy(),
    "real": test["y"].to_numpy(),
    "mean": forecast_mean["y_hat"].to_numpy(),
    "naive": forecast_naive["y_hat"].to_numpy(),
    "drift": forecast_drift["y_hat"].to_numpy(),
})

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(train["ds"].tail(30), train["y"].tail(30), label="Train reciente")
ax.plot(test["ds"], test["y"], marker="o", linewidth=2, label="Real")
ax.plot(test["ds"], comparacion["mean"], label="Mean")
ax.plot(test["ds"], comparacion["naive"], label="Naive")
ax.plot(test["ds"], comparacion["drift"], label="Drift")

ax.set(
    title="Benchmarks sobre el conjunto de prueba",
    xlabel="Fecha",
    ylabel="Pasajeros"
)
ax.legend()
plt.show()


### 🔎 Interpreta

Sin calcular todavía métricas:

1. ¿qué método parece aproximarse mejor a `test`?
2. ¿mean y naive producen trayectorias horizontales?
3. ¿drift cambia con el horizonte?
4. ¿la inspección visual basta para declarar un ganador?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

La gráfica orienta, pero no basta. Necesitamos métricas calculadas sobre exactamente las mismas observaciones de prueba.

</details>


## 7. Error de pronóstico

Definimos:

$$
e_t=y_t-\hat{y}_t
$$

Un error positivo significa que el pronóstico quedó por debajo del valor real.


In [ ]:
comparacion["error_mean"] = comparacion["real"] - comparacion["mean"]
comparacion["error_naive"] = comparacion["real"] - comparacion["naive"]
comparacion["error_drift"] = comparacion["real"] - comparacion["drift"]

comparacion[
    [
        "ds",
        "real",
        "mean",
        "error_mean",
        "naive",
        "error_naive",
        "drift",
        "error_drift",
    ]
].head()


## 8. MAE

El **Mean Absolute Error** es:

$$
MAE=\frac{1}{n}\sum_{t=1}^{n}|e_t|
$$

Se expresa en las mismas unidades de la variable.


In [ ]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred))


## 9. RMSE

El **Root Mean Squared Error** es:

$$
RMSE=
\sqrt{
\frac{1}{n}\sum_{t=1}^{n}e_t^2
}
$$

Penaliza con mayor intensidad los errores grandes.


In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


### 🧠 Predice

Si un método tiene muchos errores pequeños pero uno extremadamente grande, ¿qué métrica tenderá a penalizarlo más: MAE o RMSE?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

RMSE, porque eleva los errores al cuadrado antes de promediarlos.

</details>


## 10. MASE

Para una serie no estacional:

$$
MASE=
\frac{
\text{MAE del pronóstico}
}{
\frac{1}{T-1}\sum_{t=2}^{T}|y_t-y_{t-1}|
}
$$

Interpretación:

- `MASE < 1`: mejor que la escala naive usada como referencia;
- `MASE > 1`: peor que esa referencia;
- `MASE = 1`: desempeño equivalente en esa escala.

La escala debe calcularse **solo con train**.


In [ ]:
def mase(y_train, y_true, y_pred):
    y_train = np.asarray(y_train)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    escala = np.mean(np.abs(np.diff(y_train)))

    if np.isclose(escala, 0):
        return np.nan

    return np.mean(np.abs(y_true - y_pred)) / escala


## 11. Tabla de evaluación


In [ ]:
resultados = []

for metodo in ["mean", "naive", "drift"]:
    resultados.append({
        "metodo": metodo,
        "MAE": mae(comparacion["real"], comparacion[metodo]),
        "RMSE": rmse(comparacion["real"], comparacion[metodo]),
        "MASE": mase(
            train["y"],
            comparacion["real"],
            comparacion[metodo]
        )
    })

metricas = (
    pd.DataFrame(resultados)
    .sort_values("MAE")
    .reset_index(drop=True)
)

metricas


### 🔎 Interpreta

1. ¿qué método obtiene el menor MAE?
2. ¿el mismo método obtiene el menor RMSE?
3. ¿qué información adicional aporta MASE?
4. ¿por qué todos deben evaluarse sobre el mismo `test`?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

MAE mide el error promedio en las unidades de la serie. RMSE penaliza más los errores grandes. MASE expresa el error en una escala relativa a una referencia naive calculada con el entrenamiento. La comparación solo es justa si todos enfrentan las mismas observaciones futuras.

</details>


## 12. ¿Qué benchmark conservar?

Un benchmark establece una **línea base**.

Más adelante, al ajustar ETS o ARIMA, preguntaremos:

> ¿El modelo más complejo mejora realmente un benchmark simple fuera de muestra?

Si no lo hace, la complejidad adicional necesita una justificación.


### 🛠️ Modifica

Cambie el horizonte a:

```python
h = 20
```

y repita el análisis.

Observe si cambia:

- el ranking entre mean, naive y drift;
- el nivel de error;
- la elección del benchmark.

Reflexione:

> ¿Puede declararse que un método es “el mejor” sin especificar horizonte, partición y métrica?


## 13. Nota sobre seasonal naive

El método **seasonal naive** utiliza como pronóstico el valor observado en la misma posición del ciclo estacional anterior.

Por ejemplo, para una serie mensual con estacionalidad anual:

$$
\hat{y}_{T+h|T}=y_{T+h-12}
$$

Es un benchmark importante cuando existe estacionalidad clara.

No debe utilizarse automáticamente solo porque una serie sea semanal, mensual o diaria. En los proyectos será opcional y deberá justificarse a partir del Hito 1.


## 14. Conexión con el proyecto integrador

En el **Hito 2** cada equipo deberá establecer la referencia contra la cual evaluará los modelos posteriores.

Para su $Y_t$ deberán definir:

1. horizonte de pronóstico;
2. partición train/test;
3. benchmarks pertinentes;
4. métricas de evaluación;
5. benchmark principal.

Todavía **no se selecciona el modelo final**.

**Hito 1:** comprender la estructura temporal.  
**Hito 2:** establecer cómo evaluar y qué benchmark superar.  
**Hito 3:** evaluar ETS.  
**Hito 4:** evaluar ARIMA e integrar la decisión final.


## 15. Cierre

### Ideas que deben quedar claras

- Forecasting se evalúa **fuera de muestra**.
- La partición train/test debe respetar el orden temporal.
- El horizonte $h$ forma parte del problema.
- Mean, naive y drift son benchmarks simples pero útiles.
- MAE y RMSE se expresan en las unidades de la serie.
- RMSE penaliza más los errores grandes.
- MASE expresa el error respecto a una escala naive.
- Un modelo complejo debe superar una referencia simple bajo una comparación justa.

> **La evaluación no es una etapa posterior al modelado: es parte de la definición del problema de forecasting.**
